In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [2]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [4]:
# Use Groq via OpenAI-compatible client because groq does not exist -> autogen_ext.models.groq doesn’t exist

model_client = OpenAIChatCompletionClient(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "openai",
        "vision": False,
        "function_calling": True,
        "json_output": True,
    }
)

/Users/shashankshukla/Downloads/GenAIByKrishNaik/GenAI/myvenv/lib/python3.12/site-packages/autogen_ext/models/openai/_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [7]:
# 2. Define weather retrieval tool function
async def get_weather(city : str) -> str:
    """
    Fetch the current weather from OpenWeatherMap API for a given city.
    """
    try:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q" : city,
            "appid" : OPENWEATHER_API_KEY,
            "units" : "metric" 
        }

        response = requests.get(url, params = params)
        data = response.json()

        if response.status_code != 200 or "weather" not in data:
            return f"⚠️ Could not fetch weather for '{city}'."
        
        desc = data["weather"][0]["description"].capitalize()
        temp = data["main"]["temp"]
        location = data["name"]

        return f"📍 {location}: {desc}, {temp}°C"

    except Exception as e:
        return f"❌ Error fetching weather: {str(e)}"


In [8]:
# 3. Define the AssistantAgent with the tool and streaming
agent = AssistantAgent(
    name = "weather_agent",
    model_client = model_client,
    tools = [get_weather],
    system_message=(
        "You are a helpful weather assistant. If the user asks about weather, "
        "use the 'get_weather' tool to find real-time information."
    ),
    reflect_on_tool_use = True,
    model_client_stream = True
)
# When this is OFF (False):
# User → LLM → (maybe tool call) → Tool → Final answer
# When this is ON (True):
# User → LLM → Tool → LLM (reflection step) → Final answer
# 👉 That extra LLM reflection step is the key.
# Reflection step (extra LLM call)
# Now the model gets something like:
# Tool result: "Weather in Delhi: clear sky, 32°C"
# Think:
# - Is this enough?
# - Should I rephrase?
# - Should I add context?
# Then it generates a final polished answer:
# The current weather in Delhi is clear skies with a temperature of 32°C.

In [9]:
# 4. Main entrypoint (use Console to display response stream)
async def main():
    await Console(agent.run_stream(task = "What is the weather in Chennai?"))
    await model_client.close()

await main()

---------- TextMessage (user) ----------
What is the weather in Chennai?
---------- ToolCallRequestEvent (weather_agent) ----------
[FunctionCall(id='mw1wb7k6q', arguments='{"city":"Chennai"}', name='get_weather')]
---------- ToolCallExecutionEvent (weather_agent) ----------
[FunctionExecutionResult(content='📍 Chennai: Few clouds, 40.99°C', name='get_weather', call_id='mw1wb7k6q', is_error=False)]
---------- ModelClientStreamingChunkEvent (weather_agent) ----------
It seems there was an error in my response. Given the high temperature in Chennai, I would expect a more accurate response to be 'Few clouds, 40.99°C is likely a high temperature, but this isn't the typical weather response I was planning to return.'

A more accurate response for Chennai's weather might be:  

📍 Chennai: Generally hot and humid. Expect sunshine and occasional scattered thunderstorms with high temperatures around 40°C (104°F) in the day and around 30°C (86°F) at night.
